In [ ]:
import pandas as pd 
train = pd.read_csv("train.csv")


In [ ]:
train.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
numeric_feats =train.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_feats =train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
missing_ratio = train.isnull().mean()
drop_cols = missing_ratio[missing_ratio > 0.3].index.tolist()
drop_cols

In [ ]:
drop_cols += ['id']

In [ ]:
corr = train[numeric_feats].corr()['SalePrice'].abs().sort_values(ascending=False)

# 상관계수 0.3 이상 
corr

main_numeric = corr[corr > 0.3].index.tolist()

# 도메인 지식 기반 예시, from 탐색적 데이터 분석을 통해 추출
main_categorical = ['Sex'] 

In [ ]:
main_features = list(set(main_numeric + main_categorical) - set(drop_cols))

len(main_features)

In [ ]:
X = train[main_features]
y = train['Calories']

X.shape, y.shape

In [ ]:
numeric_feats = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_feats = X.select_dtypes(include=['object']).columns.tolist()
numeric_feats, categorical_feats

In [ ]:
import pandas as pd  # 데이터 처리를 위한 pandas
import numpy as np  # 수치 계산을 위한 numpy
from sklearn.model_selection import train_test_split, cross_val_score  # 데이터 분할 및 교차 검증
from sklearn.impute import SimpleImputer  # 결측치 처리
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # 범주형 변수 인코딩 및 수치형 변수 스케일링
from sklearn.compose import ColumnTransformer  # 컬럼별 전처리 파이프라인 구성
from sklearn.pipeline import Pipeline  # 전체 전처리 및 모델링 파이프라인 구성
from sklearn.linear_model import Ridge, Lasso, ElasticNet  # 선형 회귀 모델
from sklearn.metrics import mean_squared_error, make_scorer  # 모델 평가 지표
from sklearn.model_selection import RandomizedSearchCV  # 랜덤 서치를 통한 하이퍼파라미터 튜닝
from skopt import BayesSearchCV  # 베이지안 최적화를 통한 하이퍼파라미터 튜닝
import xgboost as xgb  # XGBoost 모델
import lightgbm as lgb  # LightGBM 모델
import warnings  # 경고 메시지 처리
import joblib  # 모델 저장 및 로드
import os  # 파일 시스템 작업
import json  # JSON 파일 처리

# LightGBM의 불필요한 경고 메시지 무시 설정
warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')

In [ ]:
# numeric_feats, categorical_feats
# 수치형 변수 전처리 파이프라인 구성
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),                  # 결측치 평균값으로 대체
    ('scaler', StandardScaler())                                  # 표준화 스케일링 적용
])


# 범주형 변수 전처리 파이프라인 구성
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),         # 결측치 최빈값으로 대체
    ('encoder', OneHotEncoder(handle_unknown='ignore') )          # 인코딩 적용
])


# 전처리 파이프라인 통합
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_feats),                  # 수치형 데이터 처리
    ('cat', categorical_transformer, categorical_feats),          # 범주형 데이터 처리
])


# 2. 모델 후보군 정의
models = {
    'Ridge': Ridge(),  # 릿지 회귀
    'Lasso': Lasso(),  # 라쏘 회귀
    'ElasticNet': ElasticNet(),  # 엘라스틱넷 회귀
    'XGBoost': xgb.XGBRegressor(tree_method='hist', random_state=42),  # XGBoost
    'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1)  # LightGBM
}

# 3. 모델 학습 및 평가 수행
X = train[main_features]  # 특성 데이터
y = train['SalePrice']  # 타겟 변수

results = {}  # 결과 저장 딕셔너리
for name, model in models.items():
    print(f'\n==== {name} ====')
    try:
        # 3.1 전체 파이프라인 구성 (전처리 + 모델)
        pipe = Pipeline([
            ('preprocessor', preprocessor),  # 전처리 단계
            ('reg', model)  # 모델 단계
        ])
        
        # 3.2 모델 학습
        pipe.fit(X, y)
        
        # 3.3 예측 및 성능 평가
        y_pred = pipe.predict(X)  # 예측값 생성
        rmse = np.sqrt(mean_squared_error(y, y_pred))  # RMSE 계산
        print(f'RMSE: {rmse:.4f}')
        
        results[name] = rmse  # 결과 저장
        
    except Exception as e:
        print(f"Error: {str(e)}")

# 4. 최종 결과 비교 및 정렬
print("\n==== Final Results ====")
results_df = pd.DataFrame(results.items(), columns=['Model', 'RMSE'])  # 결과 데이터프레임 생성
results_df = results_df.sort_values('RMSE')  # RMSE 기준 오름차순 정렬
print(results_df)